In [ ]:
pip install sentence-transformers umap-learn hdbscan scikit-learn
pip install anthropic
pip install graphviz
conda install -y graphviz
apt-get install -y graphviz

In [ ]:
import numpy as np
import pandas as pd
import re
import os
from dataclasses import dataclass, field
from typing import Optional
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN, validity_index
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text as sk_text
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class TreeNode:
    label: str
    level: int = 0
    requirements: list = field(default_factory=list)
    children: list = field(default_factory=list)
    topic_words: list = field(default_factory=list)
    dbcv_score: Optional[float] = None
    noise_assigned: list = field(default_factory=list)
    noise_propagated: list = field(default_factory=list)

    @property
    def all_requirements(self):
        reqs = list(self.requirements) + list(self.noise_assigned) + list(self.noise_propagated)
        for child in self.children:
            reqs.extend(child.all_requirements)
        return reqs

    @property
    def is_terminal(self):
        return len(self.children) == 0

    def total_nodes(self):
        return 1 + sum(c.total_nodes() for c in self.children)

    def max_depth(self):
        if not self.children:
            return 0
        return 1 + max(c.max_depth() for c in self.children)

REQUIREMENTS_STOPWORDS = [
    # Requirement boilerplate, in virtually every sentence
    "shall", "system", "provide", "support", "ensure", "include",
    "use", "used", "allow", "enable", "perform", "able", "capability",
    "function", "functionality", "feature", "based", "following",
    "using", "via", "accordance", "according", "meet", "conform",
    "comply", "define", "specify", "contain", "set",
    # Quantity/degree words with no cluster signal
    "all", "each", "per", "within", "least", "minimum", "maximum",
    "must", "will", "should", "may", "can", "least", "than",
    # Generic technical containers
    "component", "module", "unit", "device", "equipment",
    "interface", "operation", "process",
]

def _split_camelcase(text):
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

def compute_ctfidf_labels(texts, labels, top_n=5):
    unique_labels = sorted(set(labels) - {-1})
    if not unique_labels:
        return {}

    extended_stop = sk_text.ENGLISH_STOP_WORDS.union(set(REQUIREMENTS_STOPWORDS))

    cluster_docs = {
        label: " ".join(_split_camelcase(texts[i])
                        for i, l in enumerate(labels) if l == label)
        for label in unique_labels
    }

    vectorizer = CountVectorizer(
        stop_words  = list(extended_stop),
        max_features= 5000,
        ngram_range = (1, 2),
        min_df      = 1,
    )
    doc_list  = [cluster_docs[l] for l in unique_labels]
    tf_matrix = vectorizer.fit_transform(doc_list).toarray()
    vocab     = vectorizer.get_feature_names_out()

    n_docs = tf_matrix.shape[0]
    df     = np.sum(tf_matrix > 0, axis=0)
    idf    = np.log(1 + n_docs / (df + 1))
    ctfidf = tf_matrix * idf

    cluster_labels = {}
    for i, label in enumerate(unique_labels):
        top_idx = ctfidf[i].argsort()[-top_n:][::-1]
        cluster_labels[label] = [vocab[idx] for idx in top_idx]
    return cluster_labels

def generate_llm_label(requirements: list, ctfidf_words: list, client,
                       max_reqs: int = 6) str:
    """
    Ask Claude Haiku to produce a clean 'DOMAIN: Topic' label for a cluster.
    Parameters
    requirements  : requirement texts in this cluster
    ctfidf_words  : top c-TF-IDF keywords (used as hint)
    client        : anthropic.Anthropic instance
    max_reqs      : max requirements sent per call (cost control)
    Returns
    str e.g. "HYDRAULIC: Pump Assembly"
    """
    sample   = requirements[:max_reqs]
    keywords = ", ".join(ctfidf_words[:5])
    prompt = f"""You are labeling a cluster of software/engineering requirements.

Top keywords (c-TF-IDF): {keywords}
Requirements in this cluster:
{chr(10).join(f'- {r}' for r in sample)}
Generate a SHORT cluster label (max 5 words) in the format:
DOMAIN: Specific Topic
Where DOMAIN is one of: IT/SOFTWARE | HYDRAULIC | ELECTRICAL/CONTROL | SAFETY | MECHANICAL | OTHER
And 'Specific Topic' is the precise subject (e.g. 'Pump Assembly', 'Authentication', 'Valve System').
Reply with ONLY the label."""

    response = client.messages.create(
        model    = "claude-haiku-4-5-20251001",
        max_tokens = 20,
        messages = [{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip()

class RecursiveSemanticDecomposition:
    """
    Recursive Coherence-Validated Semantic Decomposition.
    Parameters
    embedding_model : str
    min_cluster_size : int
    umap_n_neighbors : int
    umap_n_components : int
    umap_min_dist : float
    min_size_for_recursion : int
    cluster_selection_method : str 'leaf' or 'eom'
    ctfidf_top_n : int
    random_state : int
    use_llm_labels : bool enable Claude-generated labels
    anthropic_api_key : str
    """

    def __init__(
        self,
        embedding_model          = "BAAI/bge-large-en-v1.5",
        min_cluster_size         = 4,
        umap_n_neighbors         = 10,
        umap_n_components        = 5,
        umap_min_dist            = 0.0,
        min_size_for_recursion   = 8,
        cluster_selection_method = "leaf",
        ctfidf_top_n             = 5,
        random_state             = 42,
        use_llm_labels           = False,
        anthropic_api_key        = None,
    ):
        self.embedding_model_name     = embedding_model
        self.min_cluster_size         = min_cluster_size
        self.umap_n_neighbors         = umap_n_neighbors
        self.umap_n_components        = umap_n_components
        self.umap_min_dist            = umap_min_dist
        self.min_size_for_recursion   = min_size_for_recursion
        self.cluster_selection_method = cluster_selection_method
        self.ctfidf_top_n             = ctfidf_top_n
        self.random_state             = random_state
        self.use_llm_labels           = use_llm_labels

        print(f"Loading sentence transformer: {embedding_model}...")
        self.encoder            = SentenceTransformer(embedding_model)
        self.texts_             = None
        self.embeddings_        = None
        self.tree_              = None
        self.decomposition_log_ = []

    def fit(self, texts, root_label="Product"):
        self.texts_ = list(texts)
        n = len(self.texts_)
        self.decomposition_log_ = []

        print(f"\n{'='*60}")
        print(f"RECURSIVE SEMANTIC DECOMPOSITION")
        print(f"{'='*60}")
        print(f"Requirements:            {n}")
        print(f"Embedding model:         {self.embedding_model_name}")
        print(f"Min cluster size:        {self.min_cluster_size}")
        print(f"Cluster selection:       {self.cluster_selection_method}")
        print(f"Min size for recursion:  {self.min_size_for_recursion}")
        print(f"LLM labels:              {self.use_llm_labels}")
        print(f"{'='*60}\n")
        print("Step 1: Computing sentence embeddings...")
        self.embeddings_ = self.encoder.encode(
            self.texts_, show_progress_bar=True, convert_to_numpy=True
        )
        print(f"  Embedding shape: {self.embeddings_.shape}")

        mean_sim = cosine_similarity(self.embeddings_).mean()
        print(f"  Mean pairwise similarity: {mean_sim:.4f}")
        if mean_sim > 0.6:
            print(f"High similarity, shared syntactic patterns detected")
        print(f"\nStep 2: Starting recursive decomposition...\n")
        root = TreeNode(label=root_label, level=0)
        self._decompose(list(range(n)), root)
        self.tree_ = root
        self._print_summary()
        return root

    def _generate_labels(self, indices, labels, indent):
        """
        Always runs c-TF-IDF.
        """
        subset_texts = [self.texts_[i] for i in indices]
        ctfidf_map   = compute_ctfidf_labels(subset_texts, labels,
                                             top_n=self.ctfidf_top_n)

        if not self.use_llm_labels or self._llm_client is None:
            return ctfidf_map, ctfidf_map

        llm_map = {}
        for cluster_id in sorted(set(labels) - {-1}):
            cluster_texts = [subset_texts[i]
                             for i, l in enumerate(labels) if l == cluster_id]
            ctfidf_words  = ctfidf_map.get(cluster_id, [])
            try:
                label = generate_llm_label(cluster_texts, ctfidf_words,
                                           self._llm_client)
                llm_map[cluster_id] = label
                print(f"{indent}    [{cluster_id}] {label}  "
                      f"(keywords: {', '.join(ctfidf_words[:3])})")
            except Exception as e:
                print(f"{indent} LLM label failed ({e}), using c-TF-IDF")
                llm_map[cluster_id] = None
        return ctfidf_map, llm_map

    def _decompose(self, indices, parent_node):
        n      = len(indices)
        level  = parent_node.level
        indent = "  " * level

        print(f"{indent}[Level {level}] Decomposing '{parent_node.label}' ({n} requirements)")

        if n < self.min_size_for_recursion:
            print(f"{indent}  Terminal (below min_size_for_recursion={self.min_size_for_recursion})")
            parent_node.requirements = indices
            return

        embeddings_subset      = self.embeddings_[indices]
        effective_n_neighbors  = min(self.umap_n_neighbors, n - 1)
        effective_n_components = min(self.umap_n_components, n - 2)

        if effective_n_components < 2:
            print(f"{indent}  Terminal (too few for UMAP)")
            parent_node.requirements = indices
            return

        umap_model = UMAP(
            n_neighbors  = effective_n_neighbors,
            n_components = effective_n_components,
            min_dist     = self.umap_min_dist,
            metric       = "cosine",
            random_state = self.random_state,
        )
        E_reduced = umap_model.fit_transform(embeddings_subset).astype(np.float64)

        effective_min_cluster = min(self.min_cluster_size, max(2, n // 4))
        hdbscan_model = HDBSCAN(
            min_cluster_size         = effective_min_cluster,
            min_samples              = 1,
            metric                   = "euclidean",
            cluster_selection_method = self.cluster_selection_method,
        )
        labels = hdbscan_model.fit_predict(E_reduced)

        unique_clusters = set(labels) - {-1}
        n_clusters      = len(unique_clusters)
        n_noise         = np.sum(labels == -1)

        if n_clusters <= 1:
            print(f"{indent}  Terminal (HDBSCAN found {n_clusters} cluster(s))")
            parent_node.requirements = indices
            return

        try:
            non_noise_mask = labels != -1
            if np.sum(non_noise_mask) < 4:
                parent_node.requirements = indices
                return
            dbcv_children = validity_index(
                E_reduced[non_noise_mask], labels[non_noise_mask], metric="euclidean"
            )
            dbcv_parent = 0.0
            print(f"{indent}  HDBSCAN: {n_clusters} clusters, {n_noise} noise points")
            print(f"{indent}  DBCV: {dbcv_children:.4f} (baseline: {dbcv_parent:.4f})")

            if dbcv_children <= dbcv_parent:
                print(f"{indent} Terminal (DBCV {dbcv_children:.4f} ≤ baseline)")
                parent_node.requirements = indices
                parent_node.dbcv_score   = dbcv_children
                return
            parent_node.dbcv_score = dbcv_children

        except Exception as e:
            print(f"{indent} Terminal (DBCV failed: {e})")
            parent_node.requirements = indices
            return

        print(f"{indent} Split accepted (DBCV={dbcv_children:.4f})")

        ctfidf_labels, display_labels = self._generate_labels(indices, labels, indent)
        self.decomposition_log_.append({
            "parent"        : parent_node.label,
            "level"         : level,
            "n_requirements": n,
            "n_clusters"    : n_clusters,
            "n_noise"       : n_noise,
            "dbcv"          : dbcv_children,
            "noise_ratio"   : n_noise / n,
        })

        cluster_centroids = {}
        cluster_child_map = {}

        for cluster_id in sorted(unique_clusters):
            local_mask             = labels == cluster_id
            cluster_global_indices = [indices[i] for i, m in enumerate(local_mask) if m]
            llm_label = display_labels.get(cluster_id)
            if llm_label and self.use_llm_labels:
                child_label = llm_label
            elif cluster_id in ctfidf_labels:
                topic_str   = ", ".join(ctfidf_labels[cluster_id][:3])
                child_label = f"Cluster_{level+1}.{cluster_id} [{topic_str}]"
            else:
                child_label = f"Cluster_{level+1}.{cluster_id}"

            child_node             = TreeNode(label=child_label, level=level + 1)
            child_node.topic_words = ctfidf_labels.get(cluster_id, [])
            parent_node.children.append(child_node)
            cluster_child_map[cluster_id] = (child_node, cluster_global_indices)

            cluster_embeddings            = self.embeddings_[cluster_global_indices]
            cluster_centroids[cluster_id] = np.mean(cluster_embeddings, axis=0)

        noise_local_indices  = [i for i, l in enumerate(labels) if l == -1]
        noise_global_indices = [indices[i] for i in noise_local_indices]
        assigned_count = propagated_count = 0

        for noise_global_idx in noise_global_indices:
            noise_emb   = self.embeddings_[noise_global_idx].reshape(1, -1)
            best_cluster, best_sim = None, -1

            for cid, centroid in cluster_centroids.items():
                sim = cosine_similarity(noise_emb, centroid.reshape(1, -1))[0, 0]
                if sim > best_sim:
                    best_sim, best_cluster = sim, cid

            child_node, cluster_indices = cluster_child_map[best_cluster]
            target_mean_sim = self._mean_intra_similarity(cluster_indices)

            if best_sim >= target_mean_sim:
                child_node.noise_assigned.append(noise_global_idx)
                cluster_child_map[best_cluster] = (child_node,
                                                   cluster_indices + [noise_global_idx])
                assigned_count += 1
            else:
                parent_node.noise_propagated.append(noise_global_idx)
                propagated_count += 1

        if noise_global_indices:
            print(f"{indent}  Noise: {assigned_count} assigned, {propagated_count} propagated")

        for cluster_id in sorted(unique_clusters):
            child_node, cluster_indices = cluster_child_map[cluster_id]
            self._decompose(cluster_indices, child_node)

    def _mean_intra_similarity(self, indices):
        if len(indices) < 2:
            return 0.0
        embs = self.embeddings_[indices]
        if len(indices) > 100:
            sample_idx = np.random.choice(len(indices), 100, replace=False)
            embs       = embs[sample_idx]
        sim_matrix = cosine_similarity(embs)
        n          = sim_matrix.shape[0]
        return np.mean(sim_matrix[np.triu_indices(n, k=1)])

    def _print_summary(self):
        tree              = self.tree_
        total_reqs        = len(self.texts_)
        total_propagated  = self._count_propagated(tree)
        rate              = total_propagated / total_reqs * 100

        print(f"\n{'='*60}")
        print(f"DECOMPOSITION SUMMARY")
        print(f"{'='*60}")
        print(f"Total requirements:     {total_reqs}")
        print(f"Tree nodes:             {tree.total_nodes()}")
        print(f"Maximum depth:          {tree.max_depth()}")
        print(f"Decomposition steps:    {len(self.decomposition_log_)}")
        print(f"Noise propagated:       {total_propagated} ({rate:.1f}%)")
        if rate > 30:
            print(f"High propagation rate")
        elif rate < 10:
            print(f"Low propagation rate")
        print(f"{'='*60}\n")

    def _count_propagated(self, node):
        count = len(node.noise_propagated)
        for child in node.children:
            count += self._count_propagated(child)
        return count

    def print_tree(self, node=None, prefix=""):
        if node is None:
            node = self.tree_
        direct     = len(node.requirements) + len(node.noise_assigned)
        propagated = len(node.noise_propagated)
        total      = len(node.all_requirements)
        info = f"{node.label}"
        if node.is_terminal:
            info += f"  [TERMINAL: {direct} reqs"
            if propagated > 0:
                info += f" + {propagated} propagated"
            info += "]"
        else:
            info += f"  [{total} total"
            if propagated > 0:
                info += f", {propagated} at this level"
            info += "]"
        if node.dbcv_score is not None:
            info += f"  (DBCV={node.dbcv_score:.3f})"
        print(f"{prefix}{info}")

        for i, child in enumerate(node.children):
            is_last      = i == len(node.children) - 1
            connector    = "└──" if is_last else "├──"
            child_prefix = prefix + ("    " if is_last else "│   ")
            self.print_tree(child, prefix + connector)

    def to_dataframe(self):
        if self.tree_ is None:
            raise ValueError("Run fit() first.")
        records = []
        self._collect_assignments(self.tree_, [], records)
        return pd.DataFrame(records)

    def _collect_assignments(self, node, path, records):
        current_path = path + [node.label]
        path_str     = " > ".join(current_path)

        for idx in node.requirements:
            records.append({"req_index": idx, "text": self.texts_[idx],
                            "cluster_path": path_str, "depth": node.level,
                            "terminal_cluster": node.label,
                            "assignment_type": "direct"})
        for idx in node.noise_assigned:
            records.append({"req_index": idx, "text": self.texts_[idx],
                            "cluster_path": path_str, "depth": node.level,
                            "terminal_cluster": node.label,
                            "assignment_type": "noise_assigned"})
        for idx in node.noise_propagated:
            records.append({"req_index": idx, "text": self.texts_[idx],
                            "cluster_path": path_str, "depth": node.level,
                            "terminal_cluster": node.label,
                            "assignment_type": "noise_propagated"})
        for child in node.children:
            self._collect_assignments(child, current_path, records)

    def get_diagnostics(self):
        return pd.DataFrame(self.decomposition_log_)

In [ ]:
import json
from pathlib import Path

def load_requirements_json(filepath: str) -> list:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list) or not all(isinstance(r, str) for r in data):
        raise ValueError("Expected a JSON file containing a flat list of strings.")
    print(f"Loaded {len(data)} requirements from '{path.name}'")
    return data
demo_texts = load_requirements_json(
    "data_file_path"
)

Loaded 97 requirements from 'dronology_dataset.json'


In [ ]:
pipeline = RecursiveSemanticDecomposition(
    embedding_model          = "BAAI/bge-large-en-v1.5",
    min_cluster_size         = 4,
    min_size_for_recursion   = 8,
    umap_n_neighbors         = 10,
    umap_n_components        = 5,
    umap_min_dist            = 0.0,
    cluster_selection_method = "leaf",
    ctfidf_top_n             = 5,
    random_state             = 42,
    use_llm_labels           = True,
    anthropic_api_key        = "API_key",
)
tree = pipeline.fit(demo_texts, root_label="Product Requirements")
print("\nREQUIREMENTS TREE:")
print("-" * 60)
pipeline.print_tree()

In [ ]:
results_df = pipeline.to_dataframe()
print(f"\nFLAT EXPORT ({len(results_df)} rows, first 20):")
print(results_df[["text", "cluster_path", "assignment_type"]].head(20).to_string())

diag = pipeline.get_diagnostics()
if not diag.empty:
    print("\nDECOMPOSITION LOG:")
    print(diag.to_string())
    
results_df.to_excel("clustering_results.xlsx", index=False)

In [ ]:
from collections import Counter
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    fowlkes_mallows_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

DRONOLOGY_COMPONENTS = {
    "Route Management": [
        r"route", r"waypoint", r"path", r"flight\s*plan",
        r"mission\s*plan", r"navigate", r"navigation",
    ],
    "Flight Control": [
        r"takeoff", r"take\s*off", r"land(?:ing)?", r"hover",
        r"fly\b", r"flight\s*mode", r"altitude", r"airspeed",
        r"RTL", r"return\s*to\s*launch", r"ascen", r"descen",
    ],
    "Flight Execution": [
        r"execut(?:e|ion)", r"assign(?:ed|ment)?.*(?:flight|route|mission)",
        r"(?:flight|route|mission).*assign", r"current\s*(?:flight|mission)",
    ],
    "UAV Registration & Activation": [
        r"register", r"registrat", r"activat", r"deactivat",
        r"virtual\s*(?:UAV|drone)", r"physical\s*(?:UAV|drone)",
        r"meta\s*data.*UAV", r"payload", r"wingspan",
    ],
    "GCS Middleware": [
        r"GCS(?:Middleware)?", r"ground\s*(?:control\s*)?station",
        r"middleware", r"handshake", r"connection.*(?:GCS|ground)",
    ],
    "Communication": [
        r"MAVLink", r"MavLink", r"message.*(?:send|receiv|forward)",
        r"command.*(?:send|forward)", r"communicat",
        r"(?:send|receiv).*(?:command|message)",
    ],
    "Monitoring & Health": [
        r"monitor", r"health", r"battery", r"voltage",
        r"heartbeat", r"status.*(?:report|updat)", r"telemetry",
    ],
    "Collision Avoidance & Safety": [
        r"collision", r"avoidance", r"separation", r"obstacle",
        r"safe(?:ty|guard)", r"emergency", r"hazard", r"geofence",
        r"no.fly", r"restricted\s*(?:area|zone|airspace)",
    ],
    "Simulation": [
        r"simulat", r"SITL", r"virtual\s*flight", r"emulat",
    ],
    "UI & Visualization": [
        r"\bUI\b", r"user\s*interface", r"display", r"map\b",
        r"visuali[sz]", r"panel", r"dashboard", r"screen",
        r"icon", r"marker", r"click", r"button",
    ],
    "Logging & Data": [
        r"\blog(?:ging|ged|s)?\b", r"record(?:ing)?", r"store",
        r"persist", r"database", r"JSON\s*(?:object|file)",
        r"configuration\s*param", r"log\s*file",
    ],
    "Coordinate System": [
        r"coordinate", r"WGS.84", r"longitude", r"latitude",
        r"GPS", r"position\s*reckon", r"geo(?:spatial|detic)",
    ],
    "Area Mapping & Search": [
        r"area\s*map", r"search\s*and\s*rescue", r"coverage",
        r"survey", r"scan(?:ning)?.*area", r"river\s*rescue",
    ],
}

def classify_requirement_keywords(text: str) -> str:
    """Classify a single requirement by keyword matching. Returns best component."""
    text_lower = text.lower()
    scores = {}
    for component, patterns in DRONOLOGY_COMPONENTS.items():
        score = sum(1 for p in patterns if re.search(p, text_lower))
        if score > 0:
            scores[component] = score

    if not scores:
        return "OTHER"
    return max(scores, key=scores.get)

def generate_ground_truth_keywords(texts: list) -> list:
    """Generate ground truth labels for all requirements using keywords."""
    labels = [classify_requirement_keywords(t) for t in texts]
    dist = Counter(labels)
    print("Ground Truth Distribution (keyword-based):")
    print("-" * 45)
    for comp, count in sorted(dist.items(), key=lambda x: -x[1]):
        print(f"  {comp:35s} {count:4d}")
    print(f"  {'TOTAL':35s} {len(labels):4d}")
    n_other = dist.get("OTHER", 0)
    if n_other > 0:
        print(f"\n {n_other} requirements classified as OTHER "
              f"({n_other/len(labels)*100:.1f}%) inspect these manually")
    return labels

def generate_ground_truth_llm(texts: list, api_key: str = None,
                              batch_size: int = 10) -> list:
    """
    Use Claude to classify each requirement into a Dronology component.
    More accurate than keywords but costs ~$0.10-0.30 for 200 requirements.
    """
    import anthropic
    import json

    client = anthropic.Anthropic(api_key=api_key)
    components = list(DRONOLOGY_COMPONENTS.keys()) + ["OTHER"]
    comp_list = "\n".join(f"- {c}" for c in components)

    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        numbered = "\n".join(f"{j+1}. {t}" for j, t in enumerate(batch))

        prompt = f"""Classify each requirement into exactly ONE Dronology system component.

Valid components:
{comp_list}

Requirements:
{numbered}

Reply with ONLY a JSON array of component names, one per requirement.
Example: ["Route Management", "UI & Visualization", "Communication"]"""

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text.strip()

        try:
            batch_labels = json.loads(raw)
            # Validate
            batch_labels = [l if l in components else "OTHER" for l in batch_labels]
        except json.JSONDecodeError:
            print(f" JSON parse failed for batch {i//batch_size}, falling back to keywords")
            batch_labels = [classify_requirement_keywords(t) for t in batch]

        all_labels.extend(batch_labels)
        print(f"  Classified {min(i+batch_size, len(texts))}/{len(texts)}")

    dist = Counter(all_labels)
    print("\nGround Truth Distribution (LLM-based):")
    print("-" * 45)
    for comp, count in sorted(dist.items(), key=lambda x: -x[1]):
        print(f"  {comp:35s} {count:4d}")
    return all_labels

def evaluate_clustering(results_df: pd.DataFrame, y_true: list,
                        save_prefix: str = "eval"):
    """
    Full external validation of clustering results against ground truth.
    Parameters
    results_df : DataFrame from pipeline.to_dataframe()
    y_true     : list of ground-truth component labels (same order as demo_texts)
    save_prefix: prefix for saved figures
    """

    y_true_arr = np.array(y_true)
    y_true_aligned = [y_true_arr[idx] for idx in results_df["req_index"]]
    y_pred = results_df["terminal_cluster"].values

    ari  = adjusted_rand_score(y_true_aligned, y_pred)
    nmi  = normalized_mutual_info_score(y_true_aligned, y_pred)
    v    = v_measure_score(y_true_aligned, y_pred)
    homo = homogeneity_score(y_true_aligned, y_pred)
    comp = completeness_score(y_true_aligned, y_pred)
    fmi  = fowlkes_mallows_score(y_true_aligned, y_pred)

    print("\n" + "=" * 60)
    print("EXTERNAL CLUSTER VALIDATION")
    print("=" * 60)
    print(f"  Adjusted Rand Index (ARI):      {ari:.4f}   (chance=0, perfect=1)")
    print(f"  Normalized Mutual Info (NMI):   {nmi:.4f}   (0=random, 1=perfect)")
    print(f"  V-measure:                      {v:.4f}   (harmonic mean of H & C)")
    print(f"  Homogeneity:                    {homo:.4f}  (each cluster = 1 GT class?)")
    print(f"  Completeness:                   {comp:.4f}  (each GT class in 1 cluster?)")
    print(f"  Fowlkes-Mallows Index:          {fmi:.4f}")
    print("=" * 60)

    print("\nInterpretation:")
    if homo > comp + 0.1:
        print("Homogeneity >> Completeness: clusters are PURE but OVER-SPLIT")
        print("(your 15 clusters subdivide the GT categories, may be desired)")
    elif comp > homo + 0.1:
        print("Completeness >> Homogeneity: clusters MERGE distinct GT categories")
        print("(some clusters contain mixed topics)")
    else:
        print("Homogeneity Completeness: balanced split/merge behavior")

    if ari > 0.6:
        print("Strong agreement with ground truth (ARI > 0.6)")
    elif ari > 0.3:
        print("Moderate agreement (ARI 0.3–0.6)")
    else:
        print("Weak agreement (ARI < 0.3) clusters differ substantially from GT")

    gt_labels_unique = sorted(set(y_true_aligned))
    pred_labels_unique = sorted(set(y_pred), key=str)
    ct = pd.crosstab(
        pd.Categorical(y_true_aligned, categories=gt_labels_unique),
        pd.Categorical(y_pred, categories=pred_labels_unique),
        rownames=["Ground Truth"],
        colnames=["Predicted Cluster"],
    )

    fig, ax = plt.subplots(figsize=(max(14, len(pred_labels_unique) * 0.9),
                                     max(6, len(gt_labels_unique) * 0.5)))
    im = ax.imshow(ct.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(pred_labels_unique)))
    short_pred = [str(l)[:30] if len(str(l)) > 30 else str(l)
                  for l in pred_labels_unique]
    ax.set_xticklabels(short_pred, rotation=75, ha="right", fontsize=7)
    ax.set_yticks(range(len(gt_labels_unique)))
    ax.set_yticklabels(gt_labels_unique, fontsize=9)

    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            val = ct.values[i, j]
            if val > 0:
                color = "white" if val > ct.values.max() * 0.5 else "black"
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=7, color=color)

    ax.set_title(f"Ground Truth vs. Clusters  (ARI={ari:.3f}, NMI={nmi:.3f}, V={v:.3f})",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Cluster (your decomposition)")
    ax.set_ylabel("Ground Truth Component")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nSaved: {save_prefix}_confusion_matrix.png")
    print("\nPer-Cluster Purity:")
    print("-" * 70)
    print(f"  {'Cluster':<35s} {'Size':>5s} {'Purity':>7s} {'Dominant GT':<20s}")
    print("-" * 70)

    for cluster in pred_labels_unique:
        mask = np.array(y_pred) == cluster
        gt_in_cluster = np.array(y_true_aligned)[mask]
        if len(gt_in_cluster) == 0:
            continue
        most_common = Counter(gt_in_cluster).most_common(1)[0]
        purity = most_common[1] / len(gt_in_cluster)
        short_name = str(cluster)[:33]
        print(f"  {short_name:<35s} {len(gt_in_cluster):5d} {purity:7.1%} {most_common[0]}")
    return {
        "ARI": ari, "NMI": nmi, "V-measure": v,
        "Homogeneity": homo, "Completeness": comp,
        "Fowlkes-Mallows": fmi,
        "n_gt_classes": len(gt_labels_unique),
        "n_pred_clusters": len(pred_labels_unique),
    }

if __name__ == "__main__":
    import json
    from pathlib import Path
    data_path = "data_file_path"
    if Path(data_path).exists():
        with open(data_path) as f:
            demo_texts = json.load(f)
    else:
        print(f"Data file not found at {data_path}")
        print("Set demo_texts and results_df from your pipeline, then call:")
        print("y_true = generate_ground_truth_keywords(demo_texts)")
        print("metrics = evaluate_clustering(results_df, y_true)")
        exit()
    print("=" * 60)
    print("GENERATING GROUND TRUTH (keyword heuristics)")
    print("=" * 60)
    y_true = generate_ground_truth_keywords(demo_texts)

In [ ]:
y_true = generate_ground_truth_llm(demo_texts, api_key="API-key")
metrics = evaluate_clustering(results_df, y_true, save_prefix="dronology_eval")